In [1]:
"""
=============================================================
PROPHET FEATURE ENGINEERING + DATASET ENRICHMENT
=============================================================
Adds 5 new temporal stress features to the ML dataset.

With only 10 days, full Prophet seasonality is unreliable.
Instead we use:
  1. Prophet trend component (fitted on available data)
  2. Is_Stress_Day (daily mean > historical mean)
  3. Day_Stress_Rank (rank of day by avg inactive, 1=worst)
  4. Hour_Zscore_Inactive (z-score of each hour vs all days)
  5. Hour_Zscore_Fluct (z-score of each hour fluctuation)

When more data is collected (60+ days), Prophet weekly and
yearly seasonality components will activate automatically.

Run this BEFORE running any model training script.
Output: ./ml_outputs_enriched/ (replaces ml_outputs for training)
=============================================================
"""

import os
import json
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

INPUT_DIR  = './ml_outputs'
OUTPUT_DIR = './ml_outputs_enriched'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 62)
print("PROPHET FEATURE ENGINEERING")
print("=" * 62)

# ── Load base data ────────────────────────────────────────────
print("\nLoading data...")
df_raw = pd.read_csv('anonymized_alarm_data.csv')
df_raw['Alarm Raised Date'] = pd.to_datetime(df_raw['Alarm Raised Date'])

X_train = pd.read_csv(f'{INPUT_DIR}/X_train.csv')
X_test  = pd.read_csv(f'{INPUT_DIR}/X_test.csv')
y_train = pd.read_csv(f'{INPUT_DIR}/y_train.csv')
y_test  = pd.read_csv(f'{INPUT_DIR}/y_test.csv')

X_train['Alarm_Raised_Date'] = pd.to_datetime(X_train['Alarm_Raised_Date'])
X_test['Alarm_Raised_Date']  = pd.to_datetime(X_test['Alarm_Raised_Date'])

print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")

# ── Step 1: Daily aggregate for Prophet + stress features ─────
print("\n[1] Computing daily stress features...")

daily_stats = df_raw.groupby('Alarm Raised Date').agg(
    daily_mean_inactive = ('Total_Inactive_Time_Seconds', 'mean'),
    daily_median_inactive = ('Total_Inactive_Time_Seconds', 'median'),
    daily_mean_fluct    = ('Fluctuation_Count', 'mean'),
    daily_std_inactive  = ('Total_Inactive_Time_Seconds', 'std'),
).reset_index().sort_values('Alarm Raised Date')

# Overall historical mean (computed on train dates only to avoid leakage)
train_dates = pd.to_datetime(X_train['Alarm_Raised_Date'].unique())
train_daily = daily_stats[daily_stats['Alarm Raised Date'].isin(train_dates)]
historical_mean_inactive = train_daily['daily_mean_inactive'].mean()
historical_std_inactive  = train_daily['daily_mean_inactive'].std()
historical_mean_fluct    = train_daily['daily_mean_fluct'].mean()

print(f"  Historical mean inactive (train): {historical_mean_inactive:.0f}s")
print(f"  Historical std  inactive (train): {historical_std_inactive:.0f}s")

# Is_Stress_Day: 1 if daily mean > historical mean
daily_stats['Is_Stress_Day'] = (
    daily_stats['daily_mean_inactive'] > historical_mean_inactive
).astype(int)

# Day_Stress_Rank: rank within known dates (1=highest inactive)
# Computed on train only, test gets rank relative to train
train_means = train_daily.set_index('Alarm Raised Date')['daily_mean_inactive']
train_rank  = train_means.rank(ascending=False).astype(int)
max_rank    = len(train_rank)

def get_stress_rank(date, val):
    """For test dates, compute rank relative to train distribution."""
    if date in train_rank.index:
        return train_rank[date]
    # Interpolate: count how many train days had higher inactive
    rank = int((train_means > val).sum()) + 1
    return min(rank, max_rank)

daily_stats['Day_Stress_Rank'] = daily_stats.apply(
    lambda row: get_stress_rank(
        row['Alarm Raised Date'],
        row['daily_mean_inactive']
    ), axis=1
)

# Day_Stress_Zscore: z-score of daily mean relative to train distribution
daily_stats['Day_Stress_Zscore'] = (
    (daily_stats['daily_mean_inactive'] - historical_mean_inactive)
    / (historical_std_inactive + 1e-8)
).clip(-3, 3)

print(f"\n  Stress day assignments:")
for _, row in daily_stats.iterrows():
    flag = '⚡ STRESS' if row['Is_Stress_Day'] else '  normal'
    print(f"    {str(row['Alarm Raised Date'].date())}: "
          f"rank={row['Day_Stress_Rank']:2d}  "
          f"zscore={row['Day_Stress_Zscore']:+.2f}  {flag}")

# ── Step 2: Prophet trend feature ────────────────────────────
print("\n[2] Fitting Prophet trend component...")

try:
    from prophet import Prophet

    prophet_df = daily_stats.rename(columns={
        'Alarm Raised Date': 'ds',
        'daily_mean_inactive': 'y'
    })[['ds', 'y']].copy()

    # Fit on train dates only
    prophet_train = prophet_df[prophet_df['ds'].isin(train_dates)]

    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,   # too few weeks for reliable estimate
        daily_seasonality=False,
        changepoint_prior_scale=0.3,
        interval_width=0.95,
        uncertainty_samples=100,
    )
    m.fit(prophet_train)

    # Predict all dates (including test — this is valid, trend only)
    forecast = m.predict(prophet_df[['ds']])
    forecast = forecast.set_index('ds')

    daily_stats['Prophet_Trend']         = daily_stats['Alarm Raised Date'].map(
        forecast['trend'])
    daily_stats['Prophet_Uncertainty']   = daily_stats['Alarm Raised Date'].map(
        forecast['yhat_upper'] - forecast['yhat_lower'])
    daily_stats['Prophet_Trend_Norm']    = (
        daily_stats['Prophet_Trend'] / (historical_mean_inactive + 1e-8)
    ).clip(0, 5)

    print("  ✅ Prophet fitted successfully")
    print(f"  Trend range: {daily_stats['Prophet_Trend'].min():.0f}s "
          f"— {daily_stats['Prophet_Trend'].max():.0f}s")

    PROPHET_AVAILABLE = True

except Exception as e:
    print(f"  ⚠️  Prophet unavailable: {e}")
    print("  Using linear trend approximation instead...")

    # Fallback: simple linear trend from date ordinal
    daily_stats['date_ord'] = (
        daily_stats['Alarm Raised Date'] -
        daily_stats['Alarm Raised Date'].min()
    ).dt.days

    from numpy.polynomial import polynomial as P
    coeffs = np.polyfit(
        daily_stats['date_ord'],
        daily_stats['daily_mean_inactive'], 1
    )
    daily_stats['Prophet_Trend'] = np.polyval(
        coeffs, daily_stats['date_ord'])
    daily_stats['Prophet_Uncertainty'] = daily_stats['daily_std_inactive'].fillna(0)
    daily_stats['Prophet_Trend_Norm']  = (
        daily_stats['Prophet_Trend'] / (historical_mean_inactive + 1e-8)
    ).clip(0, 5)

    PROPHET_AVAILABLE = False

# ── Step 3: Hourly Z-score features ─────────────────────────
print("\n[3] Computing hourly z-score features...")

inactive_cols = [f'Hour_{h}_Inactive' for h in range(1, 25)]
fluct_cols    = [f'Hour_{h}_Fluctuation' for h in range(1, 25)]

# Compute mean/std per hour across ALL training rows
train_rows = df_raw[
    df_raw['Alarm Raised Date'].isin(train_dates)
]
hour_means_inactive = train_rows[inactive_cols].mean()
hour_stds_inactive  = train_rows[inactive_cols].std() + 1e-8
hour_means_fluct    = train_rows[fluct_cols].mean()
hour_stds_fluct     = train_rows[fluct_cols].std() + 1e-8

# For each row in the full dataset, compute z-score relative to
# that row's specific hours in the window
# We add: mean z-score across the active window hours

def get_window_zscores(row, start_hour_0idx, window_hours):
    """
    For a given start hour and window, compute:
    - mean z-score of inactive time across window
    - mean z-score of fluctuation across window
    """
    hours = [(start_hour_0idx + i) % 24 + 1 for i in range(window_hours)]
    z_inactive = np.mean([
        (row[f'Hour_{h}_Inactive'] - hour_means_inactive[f'Hour_{h}_Inactive'])
        / hour_stds_inactive[f'Hour_{h}_Inactive']
        for h in hours
    ])
    z_fluct = np.mean([
        (row[f'Hour_{h}_Fluctuation'] - hour_means_fluct[f'Hour_{h}_Fluctuation'])
        / hour_stds_fluct[f'Hour_{h}_Fluctuation']
        for h in hours
    ])
    return np.clip(z_inactive, -3, 3), np.clip(z_fluct, -3, 3)

print("  Z-score stats per hour (training data):")
print(f"  Inactive — mean of means: {hour_means_inactive.mean():.1f}s, "
      f"mean of stds: {hour_stds_inactive.mean():.1f}s")
print(f"  Fluct    — mean of means: {hour_means_fluct.mean():.3f}, "
      f"mean of stds: {hour_stds_fluct.mean():.3f}")

# ── Step 4: Build enriched feature set ───────────────────────
print("\n[4] Building enriched feature matrices...")

# Create date-level feature lookup
date_features = daily_stats.set_index('Alarm Raised Date')[[
    'Is_Stress_Day',
    'Day_Stress_Rank',
    'Day_Stress_Zscore',
    'Prophet_Trend',
    'Prophet_Uncertainty',
    'Prophet_Trend_Norm',
]].to_dict(orient='index')

# Save hour normalization stats for inference
hour_norm_stats = {
    'hour_means_inactive': hour_means_inactive.to_dict(),
    'hour_stds_inactive':  hour_stds_inactive.to_dict(),
    'hour_means_fluct':    hour_means_fluct.to_dict(),
    'hour_stds_fluct':     hour_stds_fluct.to_dict(),
    'historical_mean_inactive': historical_mean_inactive,
    'historical_std_inactive':  historical_std_inactive,
    'historical_mean_fluct':    historical_mean_fluct,
    'prophet_available':        PROPHET_AVAILABLE,
}
with open(f'{OUTPUT_DIR}/hour_norm_stats.json', 'w') as f:
    json.dump(hour_norm_stats, f, indent=2)

def enrich_df(X_df, df_raw_for_hours):
    """Add 7 new features to X dataframe."""
    X_new = X_df.copy()

    # Date-level features from daily stats
    for feat in ['Is_Stress_Day','Day_Stress_Rank','Day_Stress_Zscore',
                 'Prophet_Trend','Prophet_Uncertainty','Prophet_Trend_Norm']:
        X_new[feat] = X_new['Alarm_Raised_Date'].apply(
            lambda d: date_features.get(pd.Timestamp(d), {}).get(feat, 0)
        )

    # Row-level hourly z-scores
    # We need the original hourly values — they are already in X_df
    # as Hour_1_Inactive ... Hour_24_Inactive
    # Window z-scores use Start_Hour from X_df
    print(f"  Computing window z-scores for {len(X_new):,} rows...")

    window_z_inactive = []
    window_z_fluct    = []

    for _, row in X_new.iterrows():
        start_h = int(row['Start_Hour']) - 1  # convert to 0-indexed
        # Use threshold_hours to determine window (default 4 if not available)
        # We use a fixed 4-hour window for z-score computation
        z_i, z_f = get_window_zscores(row, start_h, 4)
        window_z_inactive.append(z_i)
        window_z_fluct.append(z_f)

    X_new['Window_Zscore_Inactive'] = window_z_inactive
    X_new['Window_Zscore_Fluct']    = window_z_fluct

    return X_new

print("  Enriching X_train...")
X_train_enriched = enrich_df(X_train, df_raw)
print("  Enriching X_test...")
X_test_enriched  = enrich_df(X_test, df_raw)

# ── Step 5: Save enriched datasets ───────────────────────────
print("\n[5] Saving enriched datasets...")

new_feature_cols = [
    'Is_Stress_Day', 'Day_Stress_Rank', 'Day_Stress_Zscore',
    'Prophet_Trend', 'Prophet_Uncertainty', 'Prophet_Trend_Norm',
    'Window_Zscore_Inactive', 'Window_Zscore_Fluct',
]

print(f"\n  New features added ({len(new_feature_cols)}):")
for feat in new_feature_cols:
    train_vals = X_train_enriched[feat]
    print(f"    {feat:<30} "
          f"mean={train_vals.mean():.3f}  "
          f"std={train_vals.std():.3f}  "
          f"min={train_vals.min():.3f}  "
          f"max={train_vals.max():.3f}")

X_train_enriched.to_csv(f'{OUTPUT_DIR}/X_train.csv', index=False)
X_test_enriched.to_csv(f'{OUTPUT_DIR}/X_test.csv',   index=False)
y_train.to_csv(f'{OUTPUT_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{OUTPUT_DIR}/y_test.csv',   index=False)

# Update preprocessing meta
with open(f'{INPUT_DIR}/preprocessing_meta.json') as f:
    meta = json.load(f)

ID_COLS     = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
TARGET_COLS = [c for c in y_train.columns if c not in ID_COLS]
feat_cols   = [c for c in X_train_enriched.columns if c not in ID_COLS]

meta['feature_columns']      = feat_cols
meta['new_prophet_features'] = new_feature_cols
meta['total_features']       = len(feat_cols)
meta['prophet_available']    = PROPHET_AVAILABLE

with open(f'{OUTPUT_DIR}/preprocessing_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f"\n  X_train_enriched: {X_train_enriched.shape}")
print(f"  X_test_enriched:  {X_test_enriched.shape}")
print(f"  Total features:   {len(feat_cols)} "
      f"(was 115, added {len(new_feature_cols)})")

print("\n" + "=" * 62)
print("ENRICHMENT COMPLETE")
print("=" * 62)
print(f"\n  Outputs saved to: {OUTPUT_DIR}/")
print(f"  Prophet available: {PROPHET_AVAILABLE}")
print("\n  Next: Run model training scripts using ml_outputs_enriched/")

PROPHET FEATURE ENGINEERING

Loading data...
  X_train: (201984, 118), X_test: (135120, 118)

[1] Computing daily stress features...
  Historical mean inactive (train): 5692s
  Historical std  inactive (train): 4101s

  Stress day assignments:
    2024-06-08: rank= 2  zscore=+1.00  ⚡ STRESS
    2024-06-09: rank= 1  zscore=+1.12  ⚡ STRESS
    2024-06-16: rank= 4  zscore=-0.42    normal
    2024-06-17: rank= 6  zscore=-0.98    normal
    2024-06-18: rank= 3  zscore=+0.99  ⚡ STRESS
    2024-06-19: rank= 7  zscore=-1.15    normal
    2024-06-20: rank= 5  zscore=-0.56    normal
    2024-06-22: rank= 1  zscore=+3.00  ⚡ STRESS
    2024-06-23: rank= 5  zscore=-0.53    normal
    2024-06-24: rank= 7  zscore=-1.20    normal

[2] Fitting Prophet trend component...


Importing plotly failed. Interactive plots will not work.
11:40:49 - cmdstanpy - INFO - Chain [1] start processing
11:40:50 - cmdstanpy - INFO - Chain [1] done processing


  ✅ Prophet fitted successfully
  Trend range: 457s — 10061s

[3] Computing hourly z-score features...
  Z-score stats per hour (training data):
  Inactive — mean of means: 219.9s, mean of stds: 622.6s
  Fluct    — mean of means: 0.096, mean of stds: 0.299

[4] Building enriched feature matrices...
  Enriching X_train...
  Computing window z-scores for 201,984 rows...
  Enriching X_test...
  Computing window z-scores for 135,120 rows...

[5] Saving enriched datasets...

  New features added (8):
    Is_Stress_Day                  mean=0.346  std=0.476  min=0.000  max=1.000
    Day_Stress_Rank                mean=3.929  std=2.204  min=1.000  max=7.000
    Day_Stress_Zscore              mean=-0.101  std=0.906  min=-1.149  max=1.118
    Prophet_Trend                  mean=5824.021  std=2773.934  min=2858.191  max=10061.430
    Prophet_Uncertainty            mean=9800.972  std=554.905  min=8905.506  max=10655.019
    Prophet_Trend_Norm             mean=1.023  std=0.487  min=0.502  max=1.76